# Try Vectrix with Your Own Data

**Upload a CSV file and get instant forecasts, analysis, and diagnostics.**

### Requirements
Your CSV needs:
- A **date column** (any format: `2024-01-01`, `01/01/2024`, etc.)
- A **numeric value column** (the thing you want to forecast)

That's it. Vectrix handles everything else automatically.

---

In [ ]:
!pip install -q vectrix

## Step 1: Upload Your CSV

Run the cell below. A file upload dialog will appear — select your CSV file.

In [ ]:
import pandas as pd
import io

try:
    from google.colab import files
    print("Click 'Choose Files' to upload your CSV...")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(io.BytesIO(uploaded[filename]))
except ImportError:
    csv_path = "your_data.csv"  # <-- Change this to your file path
    df = pd.read_csv(csv_path)
    filename = csv_path

print(f"\nLoaded: {filename}")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
df.head()

## Step 2: Identify Your Columns

Set the date and value column names from your CSV. Look at the output above to find the right names.

In [ ]:
#  EDIT THESE TWO LINES 
DATE_COL = "date"       # <-- Your date column name
VALUE_COL = "value"     # <-- Your numeric value column name

# Validate
assert DATE_COL in df.columns, f"Column '{DATE_COL}' not found. Available: {list(df.columns)}"
assert VALUE_COL in df.columns, f"Column '{VALUE_COL}' not found. Available: {list(df.columns)}"
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
print(f"Date range: {df[DATE_COL].min()} to {df[DATE_COL].max()}")
print(f"Value range: {df[VALUE_COL].min():.2f} to {df[VALUE_COL].max():.2f}")
print(f"Missing values: {df[VALUE_COL].isna().sum()}")

## Step 3: Automatic Analysis

Get a complete profile of your data — trend, seasonality, anomalies, DNA fingerprint, and model recommendations.

In [ ]:
from vectrix import analyze

analysis = analyze(df, date=DATE_COL, value=VALUE_COL)
print(analysis.summary())

In [ ]:
# DNA fingerprint
dna = analysis.dna
print("=== Your Data's DNA ===")
print(f"Trend strength:      {dna.features['trendStrength']:.3f}")
print(f"Seasonal strength:   {dna.features['seasonalStrength']:.3f}")
print(f"Volatility:          {dna.features['volatility']:.3f}")
print(f"Difficulty:          {dna.difficulty}")
print(f"Dominant period:     {dna.features['seasonalPeakPeriod']:.0f}")
print(f"\nRecommended models:")
for i, m in enumerate(dna.recommendedModels[:5], 1):
    print(f"  {i}. {m}")

In [ ]:
# Anomalies (indices of outlier points)
print(f"\nAnomalies detected: {len(analysis.anomalies)}")
if len(analysis.anomalies) > 0:
    for idx in analysis.anomalies[:10]:
        print(f"  Index {idx}")

## Step 4: Forecast

Set how many steps ahead you want to predict.

In [ ]:
#  EDIT THIS LINE 
STEPS = 30  # <-- How many periods ahead to forecast

from vectrix import forecast

result = forecast(df, date=DATE_COL, value=VALUE_COL, steps=STEPS)
print(result.summary())

In [ ]:
# Forecast table with confidence intervals
result.toDataframe()

In [ ]:
# Accuracy metrics
print(f"Best model: {result.model}")
print(f"MAPE: {result.mape:.2f}%")
print(f"RMSE: {result.rmse:.2f}")
print(f"MAE:  {result.mae:.2f}")

## Step 5: Compare All Models

See how every model performs on your data.

In [ ]:
from vectrix import compare

comp = compare(df, date=DATE_COL, value=VALUE_COL, steps=STEPS)
comp

## Step 6: Quick Report (Everything at Once)

Get analysis + forecast in a single call.

In [ ]:
from vectrix import quickReport

report = quickReport(df, date=DATE_COL, value=VALUE_COL, steps=STEPS)

print("=== Quick Report ===")
print(f"\nForecast model: {report['forecast'].model}")
print(f"MAPE: {report['forecast'].mape:.2f}%")
print(f"\nData difficulty: {report['analysis'].dna.difficulty}")
print(f"Trend: {report['analysis'].dna.features['trendStrength']:.3f}")
print(f"Seasonality: {report['analysis'].dna.features['seasonalStrength']:.3f}")
print(f"Anomalies: {len(report['analysis'].anomalies)}")

## Step 7: Export Results

Download your forecast as a CSV file.

In [ ]:
# Export forecast to CSV
forecast_df = result.toDataframe()
forecast_df.to_csv("vectrix_forecast.csv", index=False)
print("Saved to vectrix_forecast.csv")

try:
    from google.colab import files
    files.download("vectrix_forecast.csv")
except ImportError:
    pass

---

## Need Help?

- **[Documentation](https://eddmpython.github.io/vectrix/docs/)** — Full guides and API reference
- **[GitHub Issues](https://github.com/eddmpython/vectrix/issues)** — Bug reports and feature requests
- **[Benchmarks](https://eddmpython.github.io/vectrix/docs/benchmarks/)** — M4 Competition results (OWA 0.877)